# 03. 선형회귀 (`nn.Module`)

02에서 손으로 구현한 경사하강법을 PyTorch의 표준 구성요소로 다시 작성합니다.

**학습 목표**
- `nn.Module`로 모델을 정의한다
- 손실함수(`nn.MSELoss`)와 옵티마이저(`optim.SGD`)를 사용한다
- 표준 학습 루프(forward → loss → backward → step)를 익힌다

데이터: `data/linear.csv` (특성 2개, $y = 3x_1 - 2x_2 + 5 + \epsilon$)

In [ ]:
import torch

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, device: {device}")

## 1. 데이터 준비

`data/linear.csv`가 없으면 합성 데이터를 만들어 저장합니다.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("data")
csv_path = DATA / "linear.csv"

if not csv_path.exists():
    rng = np.random.default_rng(0)
    x = rng.uniform(-3, 3, size=(200, 2))
    y = 3 * x[:, 0] - 2 * x[:, 1] + 5 + rng.normal(0, 0.5, size=200)
    DATA.mkdir(exist_ok=True)
    pd.DataFrame({"x1": x[:, 0], "x2": x[:, 1], "y": y}).round(4).to_csv(csv_path, index=False)
    print("생성:", csv_path)

df = pd.read_csv(csv_path)
df.head()

In [ ]:
X = torch.tensor(df[["x1", "x2"]].values, dtype=torch.float32)
y = torch.tensor(df[["y"]].values, dtype=torch.float32)

# 학습/검증 분할 (80/20)
perm = torch.randperm(len(X))
train_idx, val_idx = perm[:160], perm[160:]
X_train, y_train = X[train_idx].to(device), y[train_idx].to(device)
X_val, y_val = X[val_idx].to(device), y[val_idx].to(device)
print(X_train.shape, X_val.shape)

## 2. 모델 정의

`nn.Module`을 상속하고 `__init__`에서 레이어를, `forward`에서 계산을 정의합니다.

In [ ]:
from torch import nn

class LinearRegression(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.linear(x)

model = LinearRegression(2).to(device)
print(model)
for name, p in model.named_parameters():
    print(name, p.data)

## 3. 학습

In [ ]:
from torch import optim

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)

history = {"train": [], "val": []}
for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train), y_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val)
    history["train"].append(loss.item())
    history["val"].append(val_loss.item())
    if epoch % 40 == 0 or epoch == 199:
        print(f"epoch {epoch:3d} | train {loss.item():.4f} | val {val_loss.item():.4f}")

In [ ]:
w = model.linear.weight.data.cpu().squeeze()
b = model.linear.bias.data.cpu().item()
print(f"학습된 식: y = {w[0]:.3f}*x1 + {w[1]:.3f}*x2 + {b:.3f}")
print("정답 식  : y = 3*x1 - 2*x2 + 5")

## 4. 결과 시각화

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    pred_val = model(X_val).cpu()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["train"], label="train")
axes[0].plot(history["val"], label="val")
axes[0].set(title="Loss", xlabel="epoch", ylabel="MSE", yscale="log")
axes[0].legend()
axes[1].scatter(y_val.cpu(), pred_val, s=12)
lims = [y.min().item(), y.max().item()]
axes[1].plot(lims, lims, "r--")
axes[1].set(title="Validation: actual vs predicted", xlabel="actual", ylabel="predicted")
plt.tight_layout()
plt.show()

## 요약
- 모델은 `nn.Module`, 손실은 `nn.MSELoss`, 갱신은 `optim.SGD`가 담당한다
- 학습 루프: `zero_grad()` → forward → loss → `backward()` → `step()`
- 평가할 때는 `model.eval()`과 `torch.no_grad()`를 함께 쓴다